# Q-Mini training pipeline
From-scratch, attention-free CausalConv-MoE model. No pretrained weights are downloaded. Keep datasets and checkpoints in Drive, not this public source repository.


In [ ]:
from google.colab import drivedrive.mount('/content/drive')from pathlib import PathDATA = Path('/content/drive/MyDrive/QMini/datasets')ART = Path('/content/drive/MyDrive/QMini/artifacts')ART.mkdir(parents=True, exist_ok=True)print('Data:', DATA, '| Artifacts:', ART)

In [ ]:
import os, subprocessREPO = Path('/content/Q-Mini')if not REPO.exists(): subprocess.run(['git','clone','https://github.com/lion36508/Q-Mini.git',str(REPO)],check=True)else: subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)os.chdir(REPO)subprocess.run(['python','-m','pip','install','-e','.[dev,colab]'],check=True)import torchprint('PyTorch',torch.__version__,'CUDA',torch.cuda.is_available())if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

In [ ]:
TOK = ART/'tokenizer_152m'/'tokenizer.json'; CFG = ART/'qmini_152m_active_config.json'TOK.parent.mkdir(parents=True, exist_ok=True)if not TOK.exists():    subprocess.run([        'python','-m','qmini.cli','tokenizer',        '--data-dir',str(DATA),'--output',str(TOK),        '--vocab-size','8192','--base-config',str(REPO/'configs'/'qmini_research_152m.json'),        '--valid-fraction','0.02',        '--config-output',str(CFG)    ],check=True)else:    print('Existing tokenizer preserved:', TOK)if not CFG.is_file():    raise FileNotFoundError(        'Tokenizer exists but its active config is missing. Restore the matching config; '        'do not silently retrain or overwrite the tokenizer.'    )# Create a genuinely small smoke model but retain the exact tokenizer vocabulary.# The stock smoke config has a 128-token vocabulary and must never be paired with# the full-corpus token streams.from qmini.config import QMiniConfigbase_config = QMiniConfig.from_json(CFG)smoke_values = base_config.to_dict()smoke_values.update({    'd_model': 32,    'd_hidden': 48,    'n_layers': 2,    'n_experts': 2,    'top_k': 1,    'max_context': 64,    'kernel_size': 5,    'dilations': [1, 2],    'max_thinking_steps': 2,    'thinking_steps': 1,})smoke_config = QMiniConfig.from_dict(smoke_values)SMOKE_CFG = ART/'qmini_smoke_active_config.json'smoke_config.save(SMOKE_CFG)print('Base vocabulary:', base_config.vocab_size)print('Smoke vocabulary:', smoke_config.vocab_size)print('Smoke model parameters are computed by the runtime, not hard-coded.')

In [ ]:
TOKENS = ART/'tokens_152m_v1'required_artifacts = [    TOKENS/'train.bin',    TOKENS/'valid.bin',    TOKENS/'train.mask.bin',    TOKENS/'valid.mask.bin',    TOKENS/'dataset.json',]present = [item.is_file() for item in required_artifacts]if any(present) and not all(present):    raise RuntimeError(        'An incomplete token artifact set already exists. It has been preserved. '        'Set TOKENS to a new versioned directory such as tokens_v2; do not mix old and new streams.'    )if not all(present):    subprocess.run([        'python','-m','qmini.cli','prepare',        '--data-dir',str(DATA),'--tokenizer',str(TOK),        '--output-dir',str(TOKENS),'--valid-fraction','0.02',        '--loss-mask','auto'    ],check=True)else:    print('Existing token streams preserved:', TOKENS)# Verify that token IDs, loss masks and tokenizer all belong to one preparation run.import hashlib, jsonmanifest = json.loads((TOKENS/'dataset.json').read_text(encoding='utf-8'))def sha256_file(path):    digest = hashlib.sha256()    with path.open('rb') as handle:        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):            digest.update(chunk)    return digest.hexdigest()if manifest.get('tokenizer_sha256') != sha256_file(TOK):    raise RuntimeError('Tokenizer hash does not match token-stream manifest.')for stream_name, mask_name, manifest_key, mask_key in [    ('train.bin','train.mask.bin','train_tokens_sha256','train_mask_sha256'),    ('valid.bin','valid.mask.bin','valid_tokens_sha256','valid_mask_sha256'),]:    stream = TOKENS/stream_name    mask = TOKENS/mask_name    if mask.stat().st_size != stream.stat().st_size // 2:        raise RuntimeError(f'Loss-mask length mismatch for {stream_name}.')    if manifest.get(manifest_key) != sha256_file(stream):        raise RuntimeError(f'{stream_name} hash does not match dataset manifest.')    if manifest.get(mask_key) != sha256_file(mask):        raise RuntimeError(f'{mask_name} hash does not match dataset manifest.')print('train tokens:', (TOKENS/'train.bin').stat().st_size//2)print('valid tokens:', (TOKENS/'valid.bin').stat().st_size//2)print('train documents:', manifest['train_documents'])print('valid documents:', manifest['valid_documents'])print('loss-mask mode:', manifest['loss_mask_mode'])

In [ ]:
# Reuse a smoke run only when the training implementation, corpus,# tokenizer and model config are identical. Prior runs are never overwritten.SOURCE_COMMIT = subprocess.run(    ['git','rev-parse','HEAD'], capture_output=True, text=True, check=True).stdout.strip()implementation_files = [    REPO/'qmini'/'model.py',    REPO/'qmini'/'train.py',    REPO/'qmini'/'data.py',    REPO/'qmini'/'quantization.py',    REPO/'qmini'/'tokenizer.py',    REPO/'qmini'/'config.py',]implementation_hash = hashlib.sha256()for source_file in implementation_files:    implementation_hash.update(source_file.relative_to(REPO).as_posix().encode('utf-8'))    implementation_hash.update(source_file.read_bytes())IMPLEMENTATION_SHA256 = implementation_hash.hexdigest()smoke_identity = json.dumps({    'implementation_sha256': IMPLEMENTATION_SHA256,    'corpus_sha256': manifest['corpus_sha256'],    'tokenizer_sha256': manifest['tokenizer_sha256']['sha256'],    'config': smoke_config.to_dict(),    'training': {        'steps': 5, 'batch_size': 1, 'seq_len': 32, 'grad_accum': 1,        'learning_rate': 3e-4, 'weight_decay': 0.1, 'seed': smoke_config.seed,    },}, sort_keys=True).encode('utf-8')SMOKE_KEY = hashlib.sha256(smoke_identity).hexdigest()[:16]SMOKE = ART/'runs'/f'smoke_{SMOKE_KEY}'smoke_args = [    'python','-m','qmini.cli','train',    '--config',str(SMOKE_CFG),    '--train-bin',str(TOKENS/'train.bin'),    '--valid-bin',str(TOKENS/'valid.bin'),    '--train-mask-bin',str(TOKENS/'train.mask.bin'),    '--valid-mask-bin',str(TOKENS/'valid.mask.bin'),    '--tokenizer',str(TOK),    '--output-dir',str(SMOKE),    '--steps','5','--batch-size','1','--seq-len','32','--grad-accum','1',    '--eval-every','5','--save-every','5','--eval-batches','2']checkpoint_path = SMOKE/'latest.pt'if checkpoint_path.is_file():    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)    if checkpoint.get('config') != smoke_config.to_dict():        raise RuntimeError('Existing smoke checkpoint config differs; it was preserved.')    if int(checkpoint.get('step', 0)) < 5:        smoke_args.append('--resume')        subprocess.run(smoke_args, check=True)    else:        print('Compatible smoke run already completed:', checkpoint_path)else:    subprocess.run(smoke_args, check=True)

In [ ]:
# Version the run by training-relevant source, corpus, tokenizer, architecture# and optimizer settings. A changed setup gets a separate checkpoint directory.BASE_BATCH_SIZE = 1BASE_SEQ_LEN = base_config.max_contextBASE_GRAD_ACCUM = 16# Sparse-MoE compute-budget heuristic: 20 tokens per unique Top-1 active parameter.# This is an initial experiment target, not an established MoE scaling law or guarantee.TARGET_TOKENS = base_config.reference_tokens_at_20_per_active_parameter()TOKENS_PER_OPTIMIZER_STEP = BASE_BATCH_SIZE * BASE_SEQ_LEN * BASE_GRAD_ACCUMBASE_STEPS = (TARGET_TOKENS + TOKENS_PER_OPTIMIZER_STEP - 1) // TOKENS_PER_OPTIMIZER_STEPTRAIN_STREAM_TOKENS = int(manifest['train_tokens'])VALID_STREAM_TOKENS = int(manifest['valid_tokens'])if TRAIN_STREAM_TOKENS < BASE_SEQ_LEN + 2:    raise RuntimeError('Training stream is shorter than the configured context window.')if VALID_STREAM_TOKENS < BASE_SEQ_LEN + 2:    raise RuntimeError('Validation stream is shorter than the configured context window.')print('Total parameters:', base_config.estimated_parameter_count())print('Unique active parameters/token:', base_config.estimated_unique_active_parameter_count())print('Initial token budget:', TARGET_TOKENS)print('Optimizer steps:', BASE_STEPS)print('Approximate corpus passes:', round(TARGET_TOKENS / TRAIN_STREAM_TOKENS, 2))print('Training stream tokens:', TRAIN_STREAM_TOKENS)print('Validation stream tokens:', VALID_STREAM_TOKENS)base_identity = json.dumps({    'implementation_sha256': IMPLEMENTATION_SHA256,    'corpus_sha256': manifest['corpus_sha256'],    'tokenizer_sha256': manifest['tokenizer_sha256']['sha256'],    'config': base_config.to_dict(),    'training': {        'steps': BASE_STEPS, 'batch_size': BASE_BATCH_SIZE,        'seq_len': BASE_SEQ_LEN, 'grad_accum': BASE_GRAD_ACCUM,        'learning_rate': 3e-4, 'weight_decay': 0.1, 'seed': base_config.seed,        'target_tokens': TARGET_TOKENS,    },}, sort_keys=True).encode('utf-8')BASE_KEY = hashlib.sha256(base_identity).hexdigest()[:16]BASE = ART/'runs'/f'base_{BASE_KEY}'args = [    'python','-m','qmini.cli','train',    '--config',str(CFG),    '--train-bin',str(TOKENS/'train.bin'),    '--valid-bin',str(TOKENS/'valid.bin'),    '--train-mask-bin',str(TOKENS/'train.mask.bin'),    '--valid-mask-bin',str(TOKENS/'valid.mask.bin'),    '--tokenizer',str(TOK),    '--output-dir',str(BASE),    '--steps',str(BASE_STEPS),'--batch-size',str(BASE_BATCH_SIZE),'--seq-len',str(BASE_SEQ_LEN),'--grad-accum',str(BASE_GRAD_ACCUM),    '--eval-every','250','--save-every','250']checkpoint_path = BASE/'latest.pt'if checkpoint_path.is_file():    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)    if checkpoint.get('config') != base_config.to_dict():        raise RuntimeError('Existing base checkpoint config differs; choose a new run key.')    if int(checkpoint.get('step', 0)) < 10000:        args.append('--resume')        subprocess.run(args, check=True)    else:        print('Base training target already completed; checkpoint preserved:', checkpoint_path)else:    subprocess.run(args, check=True)EVAL_REPORT = ART/'reports'/f'language_eval_{BASE_KEY}.json'subprocess.run([    'python','-m','qmini.cli','eval-lm',    '--model-dir',str(BASE/'final_model'),    '--tokens',str(TOKENS/'valid.bin'),    '--mask',str(TOKENS/'valid.mask.bin'),    '--batch-size',str(BASE_BATCH_SIZE),'--seq-len',str(BASE_SEQ_LEN),'--batches','100',    '--output',str(EVAL_REPORT)],check=True)print('Evaluation report:', EVAL_REPORT)# Keep task-evaluation prompts and answers outside DATA so they cannot leak into# tokenizer fitting or pretraining.TASK_EVAL = ART/'evaluation'/'evaluation-tasks.jsonl'if TASK_EVAL.is_file():    TASK_REPORT = ART/'reports'/f'task_eval_{BASE_KEY}.json'    subprocess.run([        'python','-m','qmini.cli','eval-tasks',        '--model-dir',str(BASE/'final_model'),        '--dataset',str(TASK_EVAL),        '--output',str(TASK_REPORT)    ],check=True)    print('Task evaluation report:', TASK_REPORT)else:    print('Optional task set not found at', TASK_EVAL, '; task-level evaluation was skipped.')